In [7]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.model_selection import train_test_split

In [8]:
CSV = r'.\csvfiles'
PTH = r'.\picklefiles'

In [9]:
with open(f'{PTH}\\baselinedataset_trainsmall.pkl', 'rb') as f:
    dataset = pickle.load(f)

In [12]:
with open(f'{PTH}\\baselinedataset_test.pkl', 'rb') as f:
    test_dataset = pickle.load(f)

In [13]:
#Undersampling to meet class imbalances for class 0 and 1
X = dataset[:,:-1]
X_test = test_dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]
y_test = test_dataset[:, -1]
#undersample = RandomUnderSampler(sampling_strategy='majority')
#X, y = undersample.fit_resample(X, y)
#X_test, y_test = undersample.fit_resample(X_test, y_test)

In [14]:
count_neg = 0
for el in y:
    if el == 0:
        count_neg += 1

In [21]:
count_neg / len(y)

0.8481383272566567

In [15]:
count_pos = y.shape[0] - count_neg

In [16]:
scale_ratio = count_neg / count_pos

In [55]:
#Train test split
#X_train,X_test,y_train,y_test = train_test_split(X, y,test_size = 0.3, random_state = 42)
X_train = X
y_train = y
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  
    'eval_metric': 'logloss',        
    'max_depth': 3,                  
    'learning_rate': 0.1,            
    'subsample': 0.8,                
    'colsample_bytree': 0.8,         
    'seed': 42,
    'scale_pos_weight':scale_ratio #class imbalance
} 
num_rounds = 100  
model = xgb.train(params, dtrain, num_rounds)
y_pred = model.predict(dtest)
y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]  # Convert probabilities to binary predictions

print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))

Accuracy: 0.5682902909447531

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      0.61      0.70     25623
         1.0       0.15      0.34      0.20      4967

    accuracy                           0.57     30590
   macro avg       0.49      0.48      0.45     30590
weighted avg       0.72      0.57      0.62     30590



In [56]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_binary)

array([[15682,  9941],
       [ 3265,  1702]], dtype=int64)

In [57]:
import pandas as pd
test_id = pd.read_csv(f'{CSV}\\Test_ID.csv')
test_id['Predicted Label'] = y_pred_binary
test_id['Predicted Probabilities'] = y_pred

In [58]:
test_id.to_csv(f'{CSV}\\Predictions_trainsmall_XGboost.csv', index = False)

In [59]:
#Implementing Light GBM
import lightgbm as lgb
from sklearn.metrics import  roc_auc_score

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',  # for binary classification 
    'metric': 'auc', # area under the curve
    'boosting_type': 'gbdt',    # traditional Gradient Boosting Decision Tree
    'num_leaves': 31,           # number of leaves in one tree
    'learning_rate': 0.05,      # learning rate
    'feature_fraction': 0.9,    # feature fraction
    'bagging_fraction': 0.8,    # bagging fraction
    'bagging_freq': 5,          # bagging frequency
    'verbose': 0,               # 0 for silent mode
    'scale_pos_weight': scale_ratio
}
# Train the model
num_round = 100  # Number of boosting rounds
bst = lgb.train(params, train_data, num_round, valid_sets = [test_data])
# Make predictions on the test set
y_pred_prob_lgb = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_lgb = [1 if pred > 0.5 else 0 for pred in y_pred_prob_lgb]  # Convert probabilities to binary predictions

#Model evaluation 
accuracy = accuracy_score(y_test, y_pred_lgb)
roc_auc = roc_auc_score(y_test, y_pred_prob_lgb)

print(f'Accuracy on Test Set: {accuracy:.4f}')
print(f'ROC AUC on Test Set: {roc_auc:.4f}')
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.140538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[1]	valid_0's auc: 0.468654
[2]	valid_0's auc: 0.477561
[3]	valid_0's auc: 0.487107
[4]	valid_0's auc: 0.507268
[5]	valid_0's auc: 0.515888
[6]	valid_0's auc: 0.508978
[7]	valid_0's auc: 0.511108
[8]	valid_0's auc: 0.503909
[9]	valid_0's auc: 0.50762
[10]	valid_0's auc: 0.508692
[11]	valid_0's auc: 0.504913
[12]	valid_0's auc: 0.502305
[13]	valid_0's auc: 0.503155
[14]	valid_0's auc: 0.50421
[15]	valid_0's auc: 0.50521
[16]	valid_0's auc: 0.505174
[17]	valid_0's auc: 0.500897
[18]	valid_0's auc: 0.501343
[19]	valid_0's auc: 0.499084
[20]	valid_0's auc: 0.500522
[21]	valid_0's auc: 0.501887
[22]	valid_0's auc: 0.501228
[23]	valid_0's auc: 0.497313
[24]	valid_0's auc: 0.496746
[25]	valid_0's auc: 0.496961
[26]	valid_0's auc: 0.497334
[27]	valid_0's auc: 0.498401
[28]	valid_0's auc: 0.498198
[29]	valid_0's auc: 0.498167
[30]	v

In [60]:
cm = confusion_matrix(y_test, y_pred_lgb)

In [61]:
import pandas as pd
test_id = pd.read_csv(f'{CSV}\\Test_ID.csv')
test_id['Predicted Label'] = y_pred_lgb
test_id['Predicted Probabilities'] = y_pred_prob_lgb

In [62]:
test_id.to_csv(f'{CSV}\\Predictions_trainsmall_LightGBM.csv', index = False)